# 🚀 Lesson 00: RAG in 20 Lines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/blob/main/notebooks/00_rag_in_20_lines.ipynb)

> **A language model that answers from *your document*, not from a guess.**

---

## 🎯 Goal
By the end of this lesson, you will have built a complete RAG system that answers a question using only a piece of text you provide.

## 🧩 What we're building

`📄 Text` ➜ `✂️ Chunks` ➜ `🔢 Vectors` ➜ `🔍 Find the most relevant chunk` ➜ `🤖 LLM + question` ➜ `✅ Answer`

## ℹ️ Lesson info

| | |
|---|---|
| ⏱ **Time** | ~15 minutes |
| 📚 **Prerequisites** | Basic Python |
| 🔑 **API key** | Required for the chat model only (OpenRouter). Embeddings run locally. |
| 🛠 **Libraries** | `sentence-transformers`, `openai`, `numpy` |

## 💡 Note
This lesson is intentionally short and simple. The goal is to grasp the *big picture*. In later lessons we open up each step and do it properly.

## 🛠️ Step 1: Install the libraries

We only need three:

| Library | What it does |
|---|---|
| `langchain` + `langchain-openrouter` | Talks to the chat model through OpenRouter |
| `sentence-transformers` | Runs the embedding model **locally** (free, no API) |
| `numpy` | Math on vectors, to find the most similar chunk |

In [1]:
!pip install  sentence-transformers openai numpy

## 🔑 Step 2: Connect to the chat model

We use [OpenRouter](https://openrouter.ai) for the chat model. Create an API key there (**Keys** page), then run the cell below and paste it when asked.

> ⚠️ Never write your key inside the notebook. Anyone who sees the notebook would see your key.

In [2]:
from getpass import getpass
import os
from langchain.chat_models import init_chat_model

if "OPENROUTER_API_KEY" not in os.environ:
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")

> 💡 We pick a specific model on purpose. OpenRouter also offers `auto`, which chooses a model for you, but then results can change from run to run, which is bad for learning and for testing.

In [ ]:
# create chat model from free model in openrouter
model = init_chat_model(
    model="openai/gpt-4o-mini",
    model_provider="openrouter"
)

In [4]:
respone = model.invoke(
    input=[{
        "role":"user",
        "content": "Say hello in one short sentence."
    }]
)

In [5]:
print(respone.content)

Hello, I’m here to help.


## 🔢 Step 3: Load the embedding model (runs locally)

An **embedding model** turns text into a list of numbers (a *vector*). Texts with similar meaning get similar vectors, and that is how we will find the right piece of text for a question.

This model runs on your own machine, so it's free and needs no API key.

In [6]:
from sentence_transformers import SentenceTransformer

d:\AI Source\Rag_torurial\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
embedder = SentenceTransformer(
    model_name_or_path="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 25890.77it/s]


In [8]:
# --- Quick test ---
vec = embedder.encode(["hello, world!"])
print(vec.shape)

(1, 384)


## 📄 Step 4: Our document

RAG answers questions from *your* documents. To keep this lesson self-contained, we use a tiny made-up company handbook written right here in the notebook.

In [9]:
text = """Paid Time Off (PTO)
Employees with 0 to 1 years of service get 10 PTO days per year. Employees with 1 to 3 years get 15 days. Employees with more than 3 years get 20 days.

Remote Work
Employees may work remotely up to 3 days per week. Remote days must be agreed with the team lead in advance.

Equipment
The company provides one laptop per employee. Broken or lost equipment must be reported to IT within 24 hours.

Expense Reimbursement
Travel and work-related expenses are reimbursed within 30 days. Receipts must be submitted through the finance portal."""

print(len(text), "characters")

557 characters


## ✂️ Step 5: Split into chunks

We can't send a whole document to the model every time. It would be slow, expensive, and noisy. So we cut it into small pieces called **chunks**.

Here each policy is separated by a blank line, so we split on blank lines. (Real documents are messier. We'll learn better strategies in the Chunking lesson.)

In [10]:
chunks = [c.strip() for c in text.split("\n\n") if c.strip()]

print("number of chunks:", len(chunks))
print()
print(chunks[0])

number of chunks: 4

Paid Time Off (PTO)
Employees with 0 to 1 years of service get 10 PTO days per year. Employees with 1 to 3 years get 15 days. Employees with more than 3 years get 20 days.


## 🔢 Step 6: Turn every chunk into a vector

Now we use the embedding model from Step 3 on all chunks at once. Each chunk becomes one vector.

We also set `normalize_embeddings=True`. This makes every vector have length 1, so comparing two vectors later is just a dot product.

In [11]:
chunk_vector = embedder.encode(chunks, normalize_embeddings=True)

### 👀 See similarity as a number

Let's compare a question with each chunk. A **higher score means closer in meaning**.

In [12]:
question = "How many vacation days do I get after two years?"

question_vector = embedder.encode([question], normalize_embeddings=True)[0]
# similarity search
scores = chunk_vector @ question_vector

for chunk, score in zip(chunks, scores):
    print(f"{score:.3f}  |  {chunk}")
    print()

0.442  |  Paid Time Off (PTO)
Employees with 0 to 1 years of service get 10 PTO days per year. Employees with 1 to 3 years get 15 days. Employees with more than 3 years get 20 days.

0.193  |  Remote Work
Employees may work remotely up to 3 days per week. Remote days must be agreed with the team lead in advance.

0.167  |  Equipment
The company provides one laptop per employee. Broken or lost equipment must be reported to IT within 24 hours.

0.220  |  Expense Reimbursement
Travel and work-related expenses are reimbursed within 30 days. Receipts must be submitted through the finance portal.



## 🔍 Step 7: Retrieve the best chunk

The highest score wins. This step, finding the most relevant piece of text for a question, is called **retrieval**. It is the **R** in RAG.

In [13]:
import numpy as np

In [14]:
best_index = int(np.argmax(scores))
context = chunks[best_index]

print(context)

Paid Time Off (PTO)
Employees with 0 to 1 years of service get 10 PTO days per year. Employees with 1 to 3 years get 15 days. Employees with more than 3 years get 20 days.


## 🤖 Step 8: Ask the model, using only that chunk

We put the retrieved chunk and the question into one prompt, and tell the model to answer **only** from the chunk. That is the **A** and **G** in RAG: *augmented* generation.

The `system` message sets the rules. The `user` message carries the context and the question.


In [15]:
respone = model.invoke(
    input=[
        {
            "role": "system",
            "content": "Answer the question using only the context. If the answer is not in the context, say \"I don't know\"."
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]
)
print(respone.content)

After two years, you get 15 PTO days per year.


In [16]:
model = init_chat_model(
    model="openai/gpt-4o-mini",
    model_provider="openrouter"
)

respone = model.invoke(
    input=[
        {
            "role": "system",
            "content": "Answer the question using only the context. If the answer is not in the context, say \"I don't know\"."
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]
)
print(respone.content)
print(respone.response_metadata)

You get 15 vacation days after two years.
{'model_name': 'openai/gpt-4o-mini', 'id': 'gen-1793461722-hg1HzQlWCyNk2QwGSgJB', 'created': 1793461722, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 2.085e-05, 'cost_details': {'upstream_inference_completions_cost': 6e-06, 'upstream_inference_prompt_cost': 1.485e-05, 'upstream_inference_cost': 2.085e-05}, 'system_fingerprint': 'fp_2851888338'}


## 🧩 Step 9: Put it all together

Everything we did fits in one function. Given a question, it will:

1. turn the question into a vector
2. find the most similar chunk (**retrieve**)
3. send the chunk and the question to the model (**generate**)

The chunks were embedded only once, earlier. Only the question is embedded on each call.

In [18]:
def ask(question):
    question_vector = embedder.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vector @ question_vector
    best_index = int(np.argmax(scores))
    context = chunks[best_index]

    respone = model.invoke(
        input=[
            {
                "role": "system",
                "content": "Answer the question using only the context. If the answer is not in the context, say \"I don't know\"."
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }
        ]
    )
    return respone.content

## 🎉 Try it

Two questions are answered by the document. The third is **not**. A good RAG system should admit it doesn't know instead of making something up.

In [19]:
print(ask("How many vacation days do I get after two years?"))
print(ask("How many days per week can I work remotely?"))
print(ask("What is the company's dental insurance policy?"))

You get 15 vacation days after two years.
You can work remotely up to 3 days per week.
I don't know.


## 📌 Summary

You just built a complete RAG system. Each step maps to a letter of the name:

| Step | What happened | RAG letter |
|---|---|---|
| Split the document into chunks and embedded them | Prepared the knowledge | — |
| Embedded the question and found the closest chunk | **Retrieval** | **R** |
| Put that chunk into the prompt | **Augmentation** | **A** |
| The model answered from the chunk | **Generation** | **G** |

The key idea: the model doesn't need to *know* the answer. It only needs to be *shown* the right text.

## ⚠️ What this version does NOT do well

This lesson is deliberately simple. Real systems need more:

- **Chunking:** splitting on blank lines only works for tidy text.
- **Retrieval:** we take only the single best chunk. If the answer is spread over two chunks, we miss it.
- **Memory:** vectors live in a Python variable. Restart the notebook and they're gone.
- **Evaluation:** we judged the answers by eye. We have no way to *measure* quality yet.

Each of these has its own lesson later in the course.

## ✍️ Exercise (10 minutes)

1. Add a new policy to `text` (for example, **Sick Leave**: 8 days per year).
2. Re-run the cells that create `chunks` and `chunk_vectors`.
3. Ask a question that only the new policy can answer, using `ask(...)`.

**Success:** the model answers correctly from your new policy.

## 🔥 Challenge (optional)

Change `ask` so it retrieves the **top 2 chunks** instead of 1 and puts both into the prompt.

💡 Hint: look up `np.argsort`, and join the chunks with a blank line between them.

Then ask a question whose answer needs two policies at once (for example: *"Can I work remotely and still get my laptop replaced if it breaks?"*). Does the answer get better?

## 📖 Next

In the next lesson we look at the core ideas behind RAG, such as hallucination, the context window, and why not just fine-tune.

**➡️ Next lesson:** [01 Core concepts](01_core_concepts.ipynb)